# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakhawat-4/my-ml-internship-v2/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (URL)
Time window: March 2026
Why: Mid-panel month, safe for development+

One row = one page (URL)

Time window: March 2026

Why: Mid-panel month, safe for development

Code to verify:

In [4]:
import pandas as pd
import os
import subprocess

# Step 1: Clone your repo (only once)
if not os.path.exists('my-ml-internship-v2'):
    print("📥 Cloning your repo...")
    subprocess.run(['git', 'clone', 'https://github.com/Sakhawat-4/my-ml-internship-v2.git'])
    print("✅ Repo cloned!")

# Step 2: Load data
data_path = 'my-ml-internship-v2/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)

print("="*60)
print("✅ DATA LOADED SUCCESSFULLY")
print("="*60)

# Step 3: Show all columns (to find correct column names)
print("\n📋 ALL COLUMNS IN THE DATA:")
print(df.columns.tolist())

print("\n"+"="*60)
print("📊 FIRST 5 ROWS")
print("="*60)
display(df.head())

# Step 4: Check for ID column (try different names)
id_column = None
for col in df.columns:
    if 'id' in col.lower() or 'url' in col.lower() or 'page' in col.lower():
        id_column = col
        break

if id_column:
    print(f"\n✅ Found ID column: '{id_column}'")
    print(f"Total rows: {len(df)}")
    print(f"Unique {id_column}: {df[id_column].nunique()}")
    print(f"Rows == Unique? {len(df) == df[id_column].nunique()}")
else:
    print("\n⚠️ No ID column found. Using index as unique identifier.")
    print(f"Total rows: {len(df)}")

# Step 5: Availability check
print("\n"+"="*60)
print("📈 AVAILABILITY CHECK (IS TRUE)")
print("="*60)

# Find a numeric column to check availability
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
if numeric_cols:
    col_to_check = numeric_cols[0]
    available = df[df[col_to_check].notna()]
    print(f"Rows with {col_to_check}: {len(available)}")
    print(f"Availability rate: {len(available)/len(df)*100:.1f}%")

✅ DATA LOADED SUCCESSFULLY

📋 ALL COLUMNS IN THE DATA:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

📊 FIRST 5 ROWS


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



✅ Found ID column: 'content_id'
Total rows: 30000
Unique content_id: 30000
Rows == Unique? True

📈 AVAILABILITY CHECK (IS TRUE)
Rows with search_volume: 27532
Availability rate: 91.8%


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### FEATURES (used for prediction)
1. **content_score** - Available when: Page content is analyzed.
2. **page_views** - Available when: Historical view data is available.
3. **session_duration** - Available when: User session data is tracked.
4. **bounce_rate** - Available when: User behavior metrics are calculated.
5. **word_count** - Available when: Page content is parsed.

### LABEL (what we're predicting)
- **is_high_value** (1 = valuable, 0 = not valuable)
- **Proxy:** `content_score` above median value.

### CONTEXT (auxiliary information)
- **url**: Unique page identifier.
- **category**: Page category (if available).

### EXCLUDED (with reason)
- **Data after March 2026:** Excluded to prevent data leakage (future data influencing past predictions).
- **User personal data:** Not available in this anonymized dataset.+

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### FIVE FEATURES WITH "AVAILABLE WHEN?"

1. **content_score** - Available when: Page content is analyzed at ingestion time
2. **page_views** - Available when: Historical view data is aggregated daily
3. **session_duration** - Available when: User session data is tracked
4. **bounce_rate** - Available when: User interaction metrics are calculated
5. **word_count** - Available when: Page content is parsed and counted

### THE TRAP (Leakage Experiment)

**Code to show the trap:**+

In [3]:
# ============================================
# STEP 1: LOAD DATA (MUST RUN FIRST)
# ============================================
import pandas as pd
import os
import subprocess

# Clone your repo (only once)
if not os.path.exists('my-ml-internship-v2'):
    print("📥 Cloning your repo...")
    subprocess.run(['git', 'clone', 'https://github.com/Sakhawat-4/my-ml-internship-v2.git'])
    print("✅ Repo cloned!")

# Load the data
data_path = 'my-ml-internship-v2/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
print(f"✅ Data loaded! Total rows: {len(df)}")

# ============================================
# STEP 2: SHOW COLUMN NAMES
# ============================================
print("\n" + "="*60)
print("📋 ALL COLUMNS:")
print("="*60)
print(df.columns.tolist())

# ============================================
# STEP 3: FIND NUMERIC COLUMNS
# ============================================
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print("\n📊 NUMERIC COLUMNS:")
print(numeric_cols)

# ============================================
# STEP 4: LEAKAGE EXPERIMENT
# ============================================
print("\n" + "="*60)
print("THE TRAP: Leakage Experiment")
print("="*60)

# Use first 5 numeric columns as features
features = numeric_cols[:5]
print(f"Using features: {features}")

# Create label from first numeric column
first_col = numeric_cols[0]
df['is_high_value'] = df[first_col] > df[first_col].median()

# Create LEAKY feature
df['leaky_feature'] = df['is_high_value'] * 1

# Train WITH leaky feature
from sklearn.ensemble import RandomForestClassifier
X = df[['leaky_feature']]
y = df['is_high_value']
model = RandomForestClassifier()
model.fit(X, y)
print(f"🚨 SCORE with leaky feature: {model.score(X, y):.2%}")

# DELETE leaky feature
df = df.drop('leaky_feature', axis=1)
print("✅ Leaky feature deleted.")

# Train WITHOUT leaky feature
X_honest = df[features]
model.fit(X_honest, y)
print(f"✅ Honest score: {model.score(X_honest, y):.2%}")

print("\n📌 LESSON: Never use label-derived features!")



✅ Data loaded! Total rows: 30000

📋 ALL COLUMNS:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

📊 NUMERIC COLUMNS:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sess

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.